In [1]:
import urllib
import os
import zipfile
import sys
from pathlib import Path
import ssl
import torch
import numpy as np
import cv2

# **Functions**

In [2]:

NOISE_TYPE='poiss'

def add_noise(image, noise_level):
    """Add synthetic noise to image"""
    if NOISE_TYPE == 'gauss':
        noisy = image+ torch.randn_like(image) * (noise_level/255)
        return torch.clamp(noisy, 0, 1)

    if NOISE_TYPE == 'poiss':
        return torch.poisson(noise_level * image) / noise_level

    else:
      raise ValueError(f"Invalid noise type: {NOISE_TYPE}")


In [3]:
# 2. Robust Color Conversion Functions
def rgb_to_hsv_torch(image: torch.Tensor) -> torch.Tensor:
    """
    Convert RGB to HSV using PyTorch primitive operations for stability.
    """
    r, g, b = image[:, 0, :, :], image[:, 1, :, :], image[:, 2, :, :]

    max_val, _ = image.max(dim=1)
    min_val, _ = image.min(dim=1)
    diff = max_val - min_val

    # Saturation
    s = torch.zeros_like(max_val)
    mask_s = max_val > 0
    s[mask_s] = diff[mask_s] / max_val[mask_s]

    # Hue
    h = torch.zeros_like(max_val)
    mask_diff = diff > 0

    mask_r = (max_val == r) & mask_diff
    h[mask_r] = (g[mask_r] - b[mask_r]) / diff[mask_r]

    mask_g = (max_val == g) & mask_diff
    h[mask_g] = 2.0 + (b[mask_g] - r[mask_g]) / diff[mask_g]

    mask_b = (max_val == b) & mask_diff
    h[mask_b] = 4.0 + (r[mask_b] - g[mask_b]) / diff[mask_b]

    h = (h / 6.0) % 1.0

    # Value
    v = max_val

    return torch.stack([h, s, v], dim=1)

def hsv_to_rgb_torch(image: torch.Tensor) -> torch.Tensor:
    h, s, v = image[:, 0, :, :], image[:, 1, :, :], image[:, 2, :, :]

    i = (h * 6.0).floor()
    f = (h * 6.0) - i
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))

    i = i % 6

    rgb = torch.zeros_like(image)

    # Stack condition masks
    mask_0 = (i == 0)
    mask_1 = (i == 1)
    mask_2 = (i == 2)
    mask_3 = (i == 3)
    mask_4 = (i == 4)
    mask_5 = (i == 5)

    # R channel
    rgb[:, 0, :, :][mask_0] = v[mask_0]
    rgb[:, 0, :, :][mask_1] = q[mask_1]
    rgb[:, 0, :, :][mask_2] = p[mask_2]
    rgb[:, 0, :, :][mask_3] = p[mask_3]
    rgb[:, 0, :, :][mask_4] = t[mask_4]
    rgb[:, 0, :, :][mask_5] = v[mask_5]

    # G channel
    rgb[:, 1, :, :][mask_0] = t[mask_0]
    rgb[:, 1, :, :][mask_1] = v[mask_1]
    rgb[:, 1, :, :][mask_2] = v[mask_2]
    rgb[:, 1, :, :][mask_3] = q[mask_3]
    rgb[:, 1, :, :][mask_4] = p[mask_4]
    rgb[:, 1, :, :][mask_5] = p[mask_5]

    # B channel
    rgb[:, 2, :, :][mask_0] = p[mask_0]
    rgb[:, 2, :, :][mask_1] = p[mask_1]
    rgb[:, 2, :, :][mask_2] = t[mask_2]
    rgb[:, 2, :, :][mask_3] = v[mask_3]
    rgb[:, 2, :, :][mask_4] = v[mask_4]
    rgb[:, 2, :, :][mask_5] = q[mask_5]

    return torch.clamp(rgb, 0, 1)

In [4]:
def adjust_gamma_hsv(image_tensor, gamma):
    """
    Apply gamma correction only to V channel in HSV space.
    image_tensor: torch tensor, shape (1, 3, H, W), range 0-1, RGB
    """
    # 1. RGB → HSV
    hsv = rgb_to_hsv_torch(image_tensor)

    # 2. Apply gamma ONLY to V channel
    hsv[:, 2, :, :] = torch.pow(hsv[:, 2, :, :], gamma)

    # 3. HSV → RGB
    return hsv_to_rgb_torch(hsv)

# **Synthetic Dataset**


In [5]:
import glob
import random
import os
from pathlib import Path
import cv2
import torch
import numpy as np
from PIL import Image

input_folder = "./MA/dataset/Synth/Endovis2018/selected_100/*.png"
all_images = glob.glob(input_folder)
output_folder_1 = "./MA/dataset/Synth/Endovis2018/Synth_2"
output_folder_2 = "./MA/dataset/Synth/Endovis2018/Synth_3"
output_folder_3 = "./MA/dataset/Synth/Endovis2018/Synth_4"
output_folder_4 = "./MA/dataset/Synth/Endovis2018/Synth_5"

for image_path in all_images:
    original_name = os.path.basename(image_path)

    # numpy → torch, shape (1, C, H, W)
    original_np = cv2.imread(image_path)
    original = torch.from_numpy(original_np).float() / 255.0
    original = original.permute(2, 0, 1).unsqueeze(0)  # (1, C, H, W)

    for i in range(3, 6):
        print(f"  i = {i}")  # DEBUG: Check i values
        adjusted = adjust_gamma_hsv(original, i)  # your function

        for Peak in (10, 20, 30, 40, 50):
            noisy = add_noise(adjusted, Peak)  # your function

            # only convert at the end for saving
            noisy_np = (noisy.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)

            if i == 4:
                noise_folder = Path(output_folder_3) / str(Peak)
                noise_folder.mkdir(parents=True, exist_ok=True)
                
                out_name = f"{os.path.splitext(original_name)[0]}_g{i}_p{Peak}.png"
                out_path = noise_folder / out_name
                
                # Skip if already exists
                if out_path.exists():
                    print(f"Skipping {out_name} (already exists)")
                    continue
                
                cv2.imwrite(str(out_path), noisy_np)
                print(f"Saved to {Peak} folder: {out_name}")

            if i == 5:
                noise_folder = Path(output_folder_4) / str(Peak)
                noise_folder.mkdir(parents=True, exist_ok=True)
                
                out_name = f"{os.path.splitext(original_name)[0]}_g{i}_p{Peak}.png"
                out_path = noise_folder / out_name
                
                # Skip if already exists
                if out_path.exists():
                    print(f"Skipping {out_name} (already exists)")
                    continue
                
                cv2.imwrite(str(out_path), noisy_np)
                print(f"Saved to {Peak} folder: {out_name}")
"""
            if i == 3:
                # Create folder with Peak name inside output_folder_2
                noise_folder = Path(output_folder_2) / str(Peak)
                noise_folder.mkdir(parents=True, exist_ok=True)
                
                out_name = f"{os.path.splitext(original_name)[0]}_g{i}_p{Peak}.png"
                cv2.imwrite(str(noise_folder / out_name), noisy_np)
                print(f"Saved to {Peak} folder: {out_name}")
                """

  i = 3
  i = 4
Skipping seq_1_frame136_g4_p10.png (already exists)
Skipping seq_1_frame136_g4_p20.png (already exists)
Skipping seq_1_frame136_g4_p30.png (already exists)
Skipping seq_1_frame136_g4_p40.png (already exists)
Skipping seq_1_frame136_g4_p50.png (already exists)
  i = 5
Saved to 10 folder: seq_1_frame136_g5_p10.png
Saved to 20 folder: seq_1_frame136_g5_p20.png
Saved to 30 folder: seq_1_frame136_g5_p30.png
Saved to 40 folder: seq_1_frame136_g5_p40.png
Saved to 50 folder: seq_1_frame136_g5_p50.png
  i = 3
  i = 4
Skipping seq_2_frame185_g4_p10.png (already exists)
Skipping seq_2_frame185_g4_p20.png (already exists)
Skipping seq_2_frame185_g4_p30.png (already exists)
Skipping seq_2_frame185_g4_p40.png (already exists)
Skipping seq_2_frame185_g4_p50.png (already exists)
  i = 5
Saved to 10 folder: seq_2_frame185_g5_p10.png
Saved to 20 folder: seq_2_frame185_g5_p20.png
Saved to 30 folder: seq_2_frame185_g5_p30.png
Saved to 40 folder: seq_2_frame185_g5_p40.png
Saved to 50 folder

'\n            if i == 3:\n                # Create folder with Peak name inside output_folder_2\n                noise_folder = Path(output_folder_2) / str(Peak)\n                noise_folder.mkdir(parents=True, exist_ok=True)\n\n                out_name = f"{os.path.splitext(original_name)[0]}_g{i}_p{Peak}.png"\n                cv2.imwrite(str(noise_folder / out_name), noisy_np)\n                print(f"Saved to {Peak} folder: {out_name}")\n                '

In [17]:
from pathlib import Path

NOISE_LEVEL = 12
folder = Path(f'MA/dataset/Synth/Endovis2018/Synth_2/{NOISE_LEVEL}')
num_images = len(list(folder.glob('*.png')))
print(num_images)

In [20]:
from pathlib import Path
import shutil

# Define source and destination folders
source_folder = Path('./MA/dataset/Synth/Endovis2018/Synth_3')
destination_folder_15 = source_folder / '15'
destination_folder_20 = source_folder / '20' 
destination_folder_12 = source_folder / '12'

# Create all destination folders
destination_folder_15.mkdir(parents=True, exist_ok=True)
destination_folder_20.mkdir(parents=True, exist_ok=True)
destination_folder_12.mkdir(parents=True, exist_ok=True)

# Get all images
images = list(source_folder.glob('*.png')) + list(source_folder.glob('*.jpg')) + list(source_folder.glob('*.jpeg'))

# Move images based on pattern
moved_count_15 = 0
moved_count_20 = 0
moved_count_12 = 0

for img in images:
    filename = img.name
    
    if "_p15." in filename:
        dest_path = destination_folder_15 / filename
        shutil.move(str(img), str(dest_path))
        print(f"Moved to 15: {filename}")
        moved_count_15 += 1
        
    elif "_p20." in filename:  # Use elif to avoid multiple moves
        dest_path = destination_folder_20 / filename
        shutil.move(str(img), str(dest_path))
        print(f"Moved to 20: {filename}")
        moved_count_20 += 1
        
    elif "_p12." in filename:  # Use elif
        dest_path = destination_folder_12 / filename
        shutil.move(str(img), str(dest_path))
        print(f"Moved to 12: {filename}")
        moved_count_12 += 1

print(f"\nSummary:")
print(f"  Moved {moved_count_15} images to '15' folder")
print(f"  Moved {moved_count_20} images to '20' folder")
print(f"  Moved {moved_count_12} images to '12' folder")
print(f"  Total moved: {moved_count_15 + moved_count_20 + moved_count_12}")

# ***Apply to whole Image***

In [10]:
def adjust_gamma(image, gamma=1.0):
	# build a lookup table mapping the pixel values [0, 255] to
	# their adjusted gamma values
	invGamma = 1.0 / gamma
	table = np.array([((i / 255.0) ** invGamma) * 255
		for i in np.arange(0, 256)]).astype("uint8")
	# apply gamma correction using the lookup table
	return cv2.LUT(image, table)

In [7]:
import glob
import random
import os
from pathlib import Path
import cv2
import torch
import numpy as np
from PIL import Image
input_folder = "./MA/dataset/Synth/Endovis2018/selected_100/*.png"
all_images = glob.glob(input_folder)
output_folder_2 = "./MA/dataset/Synth/Endovis2018/Synth_WholeImage"

In [ ]:
for image_path in all_images:
    original_name = os.path.basename(image_path)

    # cv2 loads as numpy (H, W, C) BGR uint8 0-255
    original = cv2.imread(image_path)

    for i in range(1, 4):
        print(f"  i = {i}")
        adjusted_np = adjust_gamma(original, 1/i)  # keep as numpy

        for Peak in (12,15,20):
            # Convert to tensor inside the loop (fresh each time)
            adjusted_tensor = torch.from_numpy(adjusted_np).float() / 255.0
            adjusted_tensor = adjusted_tensor.permute(2, 0, 1).unsqueeze(0)  # (1, C, H, W)
            noisy = add_noise(adjusted_tensor, Peak)

            # Convert back to numpy for saving
            noisy_np = (noisy.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)

            if i == 2:
                noise_folder = Path(output_folder_2) / str(Peak)
                noise_folder.mkdir(parents=True, exist_ok=True)

                out_name = f"{os.path.splitext(original_name)[0]}_g{i}_p{Peak}.png"
                cv2.imwrite(str(noise_folder / out_name), noisy_np)
                print(f"Saved to {Peak} folder: {out_name}")

In [3]:
import shutil
import random
from pathlib import Path

def copy_images_across_noise_levels(
    base_folder,        # e.g., Path("MA/dataset/Synth/Endovis2018/Synth_3")
    denoised_base,      # e.g., Path("MA/dataset/Synth/Endovis2018/N2D/N2D_Colie_G3")
    gt_folder,          # Path("MA/dataset/Synth/Endovis2018/selected_100")
    output_folder,      # Where to save everything
    noise_levels,       # e.g., [50, 45, 40, ..., 10]
    num_images=3        # How many different base images to pick
):
    """
    Copy the same set of images across multiple noise levels.
    """
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    
    # Get all available base names from the highest noise level (assuming all exist)
    highest_noise = max(noise_levels)
    sample_folder = base_folder / str(highest_noise)
    all_files = list(sample_folder.glob("*_g3_p*.png"))
    
    if not all_files:
        print(f"No files found in {sample_folder}")
        return
    
    # Extract unique base names (everything before '_g')
    base_names = set()
    for f in all_files:
        base_name = f.stem.split('_g')[0]
        base_names.add(base_name)
    
    # Randomly select
    selected_bases = random.sample(sorted(base_names), min(num_images, len(base_names)))
    
    print(f"Selected base images: {selected_bases}")
    
    for base_name in selected_bases:
        print(f"\nProcessing: {base_name}")
        
        for noise in noise_levels:
            # Construct filenames
            noisy_filename = f"{base_name}_g3_p{noise}.png"
            noisy_path = base_folder / str(noise) / noisy_filename
            
            # Denoised path: denoised_base / str(noise) / filename
            denoised_path = denoised_base / str(noise) / noisy_filename
            
            # GT path (same for all noise levels)
            gt_filename = f"{base_name}.png"
            gt_path = gt_folder / gt_filename
            
            # Destination folders
            noise_out = output_folder / f"noise_{noise}" / "noisy"
            denoised_out = output_folder / f"noise_{noise}" / "denoised"
            gt_out = output_folder / "gt"
            
            noise_out.mkdir(parents=True, exist_ok=True)
            denoised_out.mkdir(parents=True, exist_ok=True)
            gt_out.mkdir(parents=True, exist_ok=True)
            
            # Check and copy
            if noisy_path.exists():
                shutil.copy2(noisy_path, noise_out / noisy_filename)
            else:
                print(f"  Missing noisy: {noisy_path}")
            
            if denoised_path.exists():
                shutil.copy2(denoised_path, denoised_out / noisy_filename)
                print(f"  Copied denoised for noise={noise}")
            else:
                print(f"  Missing denoised: {denoised_path}")
            
            # Copy GT only once
            if noise == noise_levels[0] and gt_path.exists():
                shutil.copy2(gt_path, gt_out / gt_filename)
    
    print(f"\nDone! Files copied to {output_folder}")

if __name__ == "__main__":
    noise_levels = [50, 45, 40, 35, 30, 25, 20, 15, 14, 13, 12, 11, 10]
    
    # For N2D verification
    copy_images_across_noise_levels(
        base_folder=Path("MA/dataset/Synth/Endovis2018/Synth_3"),
        denoised_base=Path("MA/dataset/Synth/Endovis2018/N2D/N2D_Colie_G3"),
        gt_folder=Path("MA/dataset/Synth/Endovis2018/selected_100"),
        output_folder=Path("MA/dataset/Synth/Endovis2018/verification_samples/N2D"),
        noise_levels=noise_levels,
        num_images=3
    )

In [6]:
import zipfile
import random
from pathlib import Path

base_path = Path("MA/dataset/Synth/Endovis2018")
output_zip = Path("MA/comparison_images.zip")

# Parameters
noise_levels = [50, 25, 10]
corruption_types = ["WholeImg", "G3"]
denoising_method = "N2D"
sci_weights = ["easy", "medium", "difficult"]
ruas_weights = ["upe", "lol", "dark"]

# Get 3 random images from first available folder (use first corruption type)
sample_folder = base_path / "SCI" / denoising_method / corruption_types[0] / sci_weights[0] / str(noise_levels[0])
print(f"Looking in: {sample_folder}")

all_images = list(sample_folder.glob("*.png"))
random.shuffle(all_images)
selected_images = all_images[:3]
print(f"Selected images: {[img.name for img in selected_images]}")

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    
    for corruption_type in corruption_types:
        for noise in noise_levels:
            
            # SCI images
            for weight in sci_weights:
                for img_path in selected_images:
                    src_path = base_path / "SCI" / denoising_method / corruption_type / weight / str(noise) / img_path.name
                    if src_path.exists():
                        arcname = f"{corruption_type}/noise_{noise}/SCI/{weight}/{img_path.name}"
                        zipf.write(src_path, arcname)
                        print(f"Added: {arcname}")
                    else:
                        print(f"Missing: {src_path}")
            
            # RUAS images
            for weight in ruas_weights:
                for img_path in selected_images:
                    src_path = base_path / "RUAS" / denoising_method / corruption_type / weight / str(noise) / img_path.name
                    if src_path.exists():
                        arcname = f"{corruption_type}/noise_{noise}/RUAS/{weight}/{img_path.name}"
                        zipf.write(src_path, arcname)
                        print(f"Added: {arcname}")
                    else:
                        print(f"Missing: {src_path}")

print(f"\nZip saved to: {output_zip}")

# ***Microscopy***

In [1]:
from skimage import io
import numpy as np
from PIL import Image
from pathlib import Path

# Load the 3D stack
img = io.imread("MA/dataset/LV200CD632s.tif")

# Create output folder
output_dir = Path("MA/dataset/Microscopy_slices")
output_dir.mkdir(parents=True, exist_ok=True)

# Loop through all slices
for i in range(img.shape[0]):
    single_slice = img[i, :, :]
    
    # Convert float32 to uint16
    slice_uint16 = (single_slice / single_slice.max() * 65535).astype(np.uint16)
    
    # Save as PNG
    Image.fromarray(slice_uint16, mode='I;16').save(output_dir / f"slice_{i:03d}.png")
    
    if i % 100 == 0:
        print(f"Saved slice {i}/{img.shape[0]}")

print(f"Saved all {img.shape[0]} slices")

/tmp/ipykernel_1735786/2688760162.py:21: DeprecationWarning: 'mode' parameter for changing data types is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(slice_uint16, mode='I;16').save(output_dir / f"slice_{i:03d}.png")


Saved slice 0/900
Saved slice 100/900
Saved slice 200/900
Saved slice 300/900
Saved slice 400/900
Saved slice 500/900
Saved slice 600/900
Saved slice 700/900
Saved slice 800/900
Saved all 900 slices


In [3]:
from pathlib import Path
import random
import numpy as np
from PIL import Image

source_dir = Path("MA/dataset/Microscopy_slices")
output_dir = Path("MA/dataset/100_microscopy")
output_dir.mkdir(parents=True, exist_ok=True)

# Get all PNG files
all_files = list(source_dir.glob("*.png"))

# Randomly select 100
selected = random.sample(all_files, 100)

# Copy and convert to uint8
for f in selected:
    img = Image.open(f)
    img_array = np.array(img)
    
    # Convert to uint8 if not already
    if img_array.dtype == np.uint16:
        img_uint8 = ((img_array - img_array.min()) / (img_array.max() - img_array.min()) * 255).astype(np.uint8)
        Image.fromarray(img_uint8).save(output_dir / f.name)
    else:
        shutil.copy(f, output_dir / f.name)

print(f"Copied and converted {len(selected)} files to {output_dir}")

Copied and converted 100 files to MA/dataset/100_microscopy
